# 12 — Vectorización ArcPy: Construcción y Bosque

Convierte el raster clasificado por el DLPK (ArcGIS Pro) en capas vectoriales limpias.  
Cumple el compromiso del **Anteproyecto §11**: generación de capas vectoriales básicas alineadas con estándares IGAC (Resolución 471/2020).

**Clases vectorizadas:** Construcción (cls=1) y Bosque (cls=2) — IoU > 0.77.

**Pipeline:**
```
Raster clasificado (ArcGIS Pro)
        |
        v
[1] PP raster a nivel mosaico completo
    MajorityFilter (8-vecinos) + BoundaryClean
        |
        v
[2] Extracción por clase (Con)
        |
        v
[3] RasterToPolygon
        |
        v
[4] Dissolve (unir adyacentes)
        |
        v
[5] SimplifyPolygon POINT_REMOVE (RDP)
        |
        v
[6] EliminatePolygonPart (huecos y esquirlas)
        |
        v
[7] CheckGeometry + RepairGeometry (OGC)
        |
        v
[8] GeoPackage (.gpkg)
```

**Prerequisito:** Correr inferencia en ArcGIS Pro con el DLPK y exportar el raster clasificado como `.tif`.

---
## 0. Configuración de rutas y parámetros

In [ ]:
import subprocess
import json
import sys
import os
from pathlib import Path

# ── Python de ArcGIS Pro ──────────────────────────────────────────────────────
ARCGIS_PYTHON = r'C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\python.exe'
SCRIPT_PATH   = str(Path('../scripts/vectorizar_arcpy.py').resolve())

# ── CONFIGURAR AQUI ───────────────────────────────────────────────────────────
# Rasters disponibles en results/inferencia_arcgis/:
#   inferencia_madrid.tif      (7.6 GB)  — tiene piramides
#   inferencia_mosquera.tif    (22  GB)  — tiene piramides
#   inferencia_tabio.tif       (1.2 GB)  — tiene piramides  <-- mas rapido para prueba
#   inferencia_facatativa.tif  (75  GB)  — SIN piramides, municipio nuevo (generalizacion)

BASE_DIR = r'D:\IGAC_1K_SEGMENTATION\IGAC_1K_SEGMENTATION'

# Municipio a procesar — cambiar segun el raster a usar
MUNICIPIO    = 'tabio'
INPUT_RASTER = rf'{BASE_DIR}\results\inferencia_arcgis\inferencia_{MUNICIPIO}.tif'

# Carpeta de salida para GeoPackages y estadisticas
OUTDIR = rf'{BASE_DIR}\results\vectores'

# Tolerancia RDP en metros (None = usar valores por clase del script)
# Construccion: 0.5m | Bosque: 1.0m
TOLERANCE = None

# Omitir PP raster si ya viene postprocesado de ArcGIS Pro
SKIP_PP = False

# ── Verificaciones ────────────────────────────────────────────────────────────
print(f'Python ArcGIS Pro : {ARCGIS_PYTHON}')
print(f'Script            : {SCRIPT_PATH}')
print(f'Input raster      : {INPUT_RASTER}')
print(f'Municipio         : {MUNICIPIO}')
print(f'Output dir        : {OUTDIR}')
print()

if not os.path.exists(ARCGIS_PYTHON):
    print('ERROR: Python de ArcGIS Pro no encontrado.')
    print(f'  Ruta buscada: {ARCGIS_PYTHON}')
    print('  Verifica la instalacion de ArcGIS Pro.')
else:
    print('Python ArcGIS Pro : OK')

if not os.path.exists(INPUT_RASTER):
    print(f'ERROR: Raster no encontrado: {INPUT_RASTER}')
    print('  Verifica que el nombre del municipio coincida con el archivo.')
    rasters = list(Path(BASE_DIR, 'results', 'inferencia_arcgis').glob('*.tif'))
    print(f'  Rasters disponibles: {[r.name for r in rasters]}')
else:
    import os
    size_gb = os.path.getsize(INPUT_RASTER) / 1e9
    print(f'Raster de entrada : OK ({size_gb:.1f} GB)')

Path(OUTDIR).mkdir(parents=True, exist_ok=True)
print(f'Carpeta de salida : OK ({OUTDIR})')

---
## 1. Ejecutar pipeline de vectorización

In [ ]:
# Construir comando
cmd = [
    ARCGIS_PYTHON,
    SCRIPT_PATH,
    '--input',     INPUT_RASTER,
    '--outdir',    OUTDIR,
    '--municipio', MUNICIPIO,
]

if TOLERANCE is not None:
    cmd += ['--tolerance', str(TOLERANCE)]

if SKIP_PP:
    cmd.append('--skip_pp')

print('Comando:', ' '.join(cmd))
print()
print('=== INICIANDO VECTORIZACION ===')
print('Tiempo estimado: 10-30 minutos segun tamano del raster.')
print()

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    encoding='utf-8',
    errors='replace'
)

# Mostrar salida en tiempo real (ya terminó, mostrar completa)
print(result.stdout)

if result.returncode != 0:
    print('=== ERRORES ===')
    print(result.stderr)
else:
    print('=== VECTORIZACION COMPLETADA EXITOSAMENTE ===')

---
## 2. Verificar archivos generados

In [ ]:
from pathlib import Path

outdir = Path(OUTDIR)
print(f'Archivos en {outdir}:')
print()

archivos_esperados = [
    f'construccion_{MUNICIPIO}.gpkg',
    f'bosque_{MUNICIPIO}.gpkg',
    f'estadisticas_vectores_{MUNICIPIO}.json',
]

for nombre in archivos_esperados:
    ruta = outdir / nombre
    if ruta.exists():
        size_mb = ruta.stat().st_size / 1e6
        print(f'  OK  {nombre}  ({size_mb:.2f} MB)')
    else:
        print(f'  FALTA  {nombre}')

# Listar todos los archivos generados
print()
print('Todos los archivos en la carpeta:')
for f in sorted(outdir.glob('*')):
    if f.is_file():
        print(f'  {f.name}  ({f.stat().st_size/1e6:.3f} MB)')

---
## 3. Estadísticas de los vectores generados

In [ ]:
import json
from pathlib import Path

json_path = Path(OUTDIR) / f'estadisticas_vectores_{MUNICIPIO}.json'

if not json_path.exists():
    print('Archivo de estadisticas no encontrado. Verifica que la vectorizacion haya completado.')
else:
    with open(json_path, encoding='utf-8') as f:
        stats = json.load(f)

    print(f'=== ESTADISTICAS DE VECTORIZACION — {MUNICIPIO.upper()} ===\n')

    for clase, s in stats.items():
        print(f'Clase: {clase} (ID={s["clase_id"]})')
        print(f'  Poligonos raw         : {s["n_poligonos_raw"]:>8,}')
        print(f'  Poligonos finales     : {s["n_poligonos_final"]:>8,}')
        print(f'  Reduccion             : {(1 - s["n_poligonos_final"]/s["n_poligonos_raw"])*100:.1f}%')
        print(f'  Errores OGC           : {s["errores_ogc"]:>8,}')
        print(f'  Area total            : {s["area_total_m2"]/1e4:>8.1f} ha')
        print(f'  Area media por polig. : {s["area_media_m2"]:>8.1f} m2')
        print(f'  Area minima           : {s["area_min_m2"]:>8.1f} m2')
        print(f'  Area maxima           : {s["area_max_m2"]/1e4:>8.2f} ha')
        print(f'  Tolerancia RDP        : {s["simplify_tol_m"]:>8.1f} m')
        print(f'  Output                : {Path(s["gpkg_path"]).name}')
        print()

---
## 4. Visualización: vectores sobre raster de predicción

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

# Intentar cargar rasterio y geopandas
try:
    import rasterio
    import geopandas as gpd
    HAS_GEO = True
except ImportError:
    HAS_GEO = False
    print('AVISO: rasterio o geopandas no disponibles.')
    print('Instalar con: pip install rasterio geopandas')

if HAS_GEO:
    import sys, os
    sys.path.insert(0, str(Path('..').resolve()))
    from scripts.utils import CLASS_NAMES, CLASS_COLORS

    N_CLASSES = 9
    CMAP = np.zeros((N_CLASSES, 3), dtype=np.uint8)
    for cls_id, rgb in CLASS_COLORS.items():
        if cls_id < N_CLASSES:
            CMAP[cls_id] = rgb

    def leer_raster_thumbnail(path, max_px=2000):
        with rasterio.open(path) as src:
            data = src.read(1)
            extent = [src.bounds.left, src.bounds.right,
                      src.bounds.bottom, src.bounds.top]
        h, w = data.shape
        scale = min(max_px / max(h, w), 1.0)
        if scale < 1.0:
            nh, nw = int(h*scale), int(w*scale)
            ri = (np.arange(nh)*h/nh).astype(int)
            ci = (np.arange(nw)*w/nw).astype(int)
            data = data[np.ix_(ri, ci)]
        return data, extent

    outdir = Path(OUTDIR)

    # Cargar raster y vectores
    raster_data, extent = leer_raster_thumbnail(INPUT_RASTER)
    raster_rgb = CMAP[np.clip(raster_data, 0, N_CLASSES-1)]

    gpkg_construccion = outdir / f'construccion_{MUNICIPIO}.gpkg'
    gpkg_bosque       = outdir / f'bosque_{MUNICIPIO}.gpkg'

    fig, axes = plt.subplots(1, 3, figsize=(22, 8), dpi=120)

    # Panel 1: Raster de prediccion
    axes[0].imshow(raster_rgb, extent=extent, aspect='equal')
    axes[0].set_title('Prediccion DeepLabV3+', fontsize=12, fontweight='bold')
    axes[0].axis('off')

    # Panel 2: Solo Construccion
    axes[1].imshow(raster_rgb, extent=extent, aspect='equal', alpha=0.4)
    if gpkg_construccion.exists():
        gdf_c = gpd.read_file(gpkg_construccion)
        gdf_c.plot(ax=axes[1], facecolor='#FF4444', edgecolor='#CC0000',
                   linewidth=0.5, alpha=0.85)
        axes[1].set_title(f'Vectores Construccion\n({len(gdf_c):,} poligonos)',
                           fontsize=12, fontweight='bold')
    else:
        axes[1].set_title('Construccion — archivo no encontrado', fontsize=11)
    axes[1].axis('off')

    # Panel 3: Solo Bosque
    axes[2].imshow(raster_rgb, extent=extent, aspect='equal', alpha=0.4)
    if gpkg_bosque.exists():
        gdf_b = gpd.read_file(gpkg_bosque)
        gdf_b.plot(ax=axes[2], facecolor='#228B22', edgecolor='#145214',
                   linewidth=0.5, alpha=0.85)
        axes[2].set_title(f'Vectores Bosque\n({len(gdf_b):,} poligonos)',
                           fontsize=12, fontweight='bold')
    else:
        axes[2].set_title('Bosque — archivo no encontrado', fontsize=11)
    axes[2].axis('off')

    plt.suptitle(f'Vectorizacion — {MUNICIPIO.capitalize()}\n'
                 f'DeepLabV3+ ResNet-50 | Construccion (IoU=0.808) + Bosque (IoU=0.771)',
                 fontsize=13, fontweight='bold', y=1.01)

    plt.tight_layout()
    fig_path = outdir / f'visualizacion_vectores_{MUNICIPIO}.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figura guardada: {fig_path}')

---
## 5. Calidad cartografica — Resolucion IGAC 471/2020

Los **4 componentes de calidad** de la norma son **criterios de aceptacion del producto**,
aplicados por el controlador de calidad (IGAC o ente validador) sobre cualquier producto
cartografico final, independientemente de si fue generado de forma manual o automatizada.

La diferencia entre cartografia manual y automatizada esta en el **proceso de produccion**,
no en el control de calidad: ambos generan un producto vectorial que debe someterse a la
misma verificacion formal por parte del receptor.

| Componente | Quien evalua | Metodo | Umbral Res. 471/2020 | Auto-verificable? |
|---|---|---|---|---|
| Consistencia logica OGC | Software | Validacion topologica | <= 3% errores | **Si** |
| Completitud / Totalidad | QC manual — muestreo | Objetos detectados vs. area proyectada | <= 3% omision | No |
| Exactitud posicional | Geodesista — puntos control | RMSE contornos vs. GT vectorial IGAC | RMSr <= 0.30 m (1:1.000) | No |
| Exactitud tematica | Fotointerprete / revisor campo | Muestrear poligonos, verificar clase | >= 95% atributos correctos | No |

> **Nota conceptual:** Los proxies de ML (IoU, Precision, Recall del test set) son
> estimaciones de cuanto podria esperar el controlador de calidad encontrar al evaluar
> el producto. No equivalen a la evaluacion formal de la norma, que requiere un proceso
> de verificacion manual externo al pipeline de produccion.
>
> **Referencia:** Instituto Geografico Agustin Codazzi — Resolucion 471 de 2020.
> Norma de referencia: ISO 19157-1:2023 — Geographic information: Data quality.

In [ ]:
import json
from pathlib import Path

# Metricas de test del modelo DeepLab (resultados definitivos del test set)
METRICAS_TEST = {
    'Construccion': {'iou': 0.808, 'precision': 0.875, 'recall': 0.914, 'f1': 0.894},
    'Bosque':       {'iou': 0.771, 'precision': 0.847, 'recall': 0.896, 'f1': 0.871},
}

json_path = Path(OUTDIR) / f'estadisticas_vectores_{MUNICIPIO}.json'
if not json_path.exists():
    print('Ejecutar primero la vectorizacion (celdas anteriores).')
else:
    with open(json_path, encoding='utf-8') as f:
        stats = json.load(f)

    print("=== EVALUACION DE CALIDAD CARTOGRAFICA ===")
    print('    Ref: Resolucion IGAC 471/2020 | ISO 19157-1:2023')
    print()
    print('NOTA: Los componentes de calidad de Res. 471/2020 son criterios de')
    print('ACEPTACION DEL PRODUCTO, evaluados por el controlador de calidad IGAC')
    print('sobre el producto entregado. Aplican igual a cartografia manual y')
    print('automatizada — la diferencia esta en el proceso de produccion, no en')
    print('el proceso de verificacion de calidad.')
    print()

    for clase, s in stats.items():
        m = METRICAS_TEST.get(clase, {})
        total_pols = s['n_poligonos_final']
        errores    = s['errores_ogc']
        pct_error  = (errores / total_pols * 100) if total_pols > 0 else 100.0

        print(f"=== {clase.upper()} ===")
        print()

        # ── 1. Consistencia logica OGC (unico auto-verificable) ────────────────
        print('  1. CONSISTENCIA LOGICA — Verificacion OGC (auto-verificable)')
        print(f'     Poligonos totales : {total_pols:,}')
        print(f'     Errores OGC       : {errores}')
        print(f'     Tasa de error     : {pct_error:.1f}%   [umbral norma: <= 3%]')
        estado_cl = 'CUMPLE' if pct_error <= 3.0 else 'NO CUMPLE — re-ejecutar tras fix Dissolve'
        print(f'     Estado            : {estado_cl}')
        print()

        # ── 2. Completitud (proceso QC manual externo) ─────────────────────────
        omision_pct = (1 - m.get('recall', 0)) * 100 if m else None
        print('  2. COMPLETITUD / TOTALIDAD — Proceso QC manual externo')
        print('     Evaluado por: controlador de calidad IGAC (muestreo sistematico)')
        print('     Metodo: verificar si objetos reales estan presentes en el producto')
        print(f'     Umbral norma   : <= 3% de omision')
        if omision_pct is not None:
            print(f'     Proxy ML (Recall test set): {m["recall"]*100:.1f}% cobertura')
            print(f'       => Omision estimada     : {omision_pct:.1f}%')
            ref = ('Supera umbral ISPRS >= 70% (Rottensteiner et al., 2014, ISPRS JPRS)'
                   if m['recall'] >= 0.70 else 'Por debajo umbral ISPRS')
            print(f'       => {ref}')
        print(f'     Area total vectorizada: {s["area_total_m2"]/1e4:.1f} ha')
        print()

        # ── 3. Exactitud posicional (proceso QC geodesico externo) ────────────
        tol = s['simplify_tol_m']
        print('  3. EXACTITUD POSICIONAL — Proceso geodesico externo')
        print('     Evaluado por: geodesista, comparando contornos vs. puntos de control')
        print(f'     Umbral norma   : RMSr <= 0.30 m (escala 1:1.000)')
        print(f'     GSD sensor     : 0.10 m  (error teorico <= 1 pixel = 0.10 m)')
        print(f'     Simplificacion : RDP tol = {tol} m (introduce desplazamiento adicional)')
        print(f'     Estimacion     : 0.10–{tol:.2f} m  — potencialmente dentro del umbral 0.30 m')
        print(f'     Estado         : Pendiente verificacion formal con GT vectorial IGAC')
        print()

        # ── 4. Exactitud tematica (proceso QC por fotointerprete) ─────────────
        print('  4. EXACTITUD TEMATICA — Proceso QC manual externo')
        print('     Evaluado por: fotointerprete o revisor de campo IGAC')
        print('     Metodo: muestrear poligonos del producto y verificar clase asignada')
        print(f'     Umbral norma   : >= 95% atributos clasificados correctamente')
        if m:
            print(f'     Proxy ML (Precision test set): {m["precision"]*100:.1f}%')
            print(f'     Proxy ML (IoU test set)      : {m["iou"]*100:.1f}%')
            ref = ('Supera umbral ISPRS >= 85% correccion (Rottensteiner et al., 2014)'
                   if m['precision'] >= 0.85 else 'Por debajo umbral ISPRS')
            print(f'     => {ref}')
        print()
        print('-' * 60)
        print()

    print('CONCLUSION:')
    print('  El pipeline produce vectores con consistencia logica OGC verificable')
    print('  automaticamente. Los tres componentes restantes requieren verificacion')
    print('  manual externa — igual que cualquier producto cartografico tradicional.')
    print('  Los proxies ML superan los umbrales de utilidad practica ISPRS para')
    print('  extraccion automatica de edificios (Rottensteiner et al., 2014).')
    print('  Se recomienda enfoque hibrido: DL produce borrador inicial, operador')
    print('  IGAC revisa y valida el resultado final.')


---
## 6. Comparacion: pipeline automatizado vs. captura manual

Estimacion de tiempo y costo para cartografiar las clases Construccion + Bosque
en el area procesada, comparando el pipeline DL con digitalizacion manual tradicional.

**Supuestos y referencias:**
- Productividad manual: **5–15 ha/dia** por operador en zonas urbanas complejas
  *(rango reportado en literatura de produccion cartografica a escala 1:1.000–1:5.000)*
  Ref: DNP (2021) *Guia de formulacion: Produccion y/o actualizacion de cartografia basica oficial*.
  URL: https://proyectostipo.dnp.gov.co/images/pdf/Cartografia/Guia_de_formulacion_PT_Cartografia_VF_20210531.pdf
- Costo operador SIG: **COP 200.000–350.000/dia** (profesional SIG junior, Colombia 2026)
  Ref: rangos salariales SENA / DANE para tecnologos y profesionales en Geomatica.
- Tiempo pipeline automatizado: estimacion conservadora (inferencia ArcGIS Pro + vectorizacion).

**Contexto de calidad del producto:**
- La comparacion de tiempo se refiere al **borrador inicial** producido por DL.
- Ambos productos (manual y DL) deben pasar el mismo control de calidad IGAC (Res. 471/2020).
- El pipeline DL supera los umbrales de utilidad practica para extraccion automatica
  (Rottensteiner et al., 2014, ISPRS JPRS: completitud >= 70%, correccion >= 85%).
- El tiempo de supervision/QA del operador sobre el resultado DL no esta incluido en el
  tiempo automatizado — se estima en 20–30% adicional del tiempo de produccion manual.

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

json_path = Path(OUTDIR) / f'estadisticas_vectores_{MUNICIPIO}.json'
if not json_path.exists():
    print('Ejecutar primero la vectorizacion.')
else:
    with open(json_path, encoding='utf-8') as f:
        stats = json.load(f)

    area_total_ha = sum(s['area_total_m2'] for s in stats.values()) / 1e4

    # -----------------------------------------------------------------------
    # Supuestos de productividad
    # Ref: DNP (2021) Guia de formulacion: Produccion y/o actualizacion de
    #      cartografia basica oficial. proyectostipo.dnp.gov.co
    # Rango: 5-15 ha/dia por operador en zonas urbanas complejas (1:1.000)
    # Valor conservador adoptado: 8 ha/dia
    # -----------------------------------------------------------------------
    PRODUCTIVIDAD_MANUAL_HA_DIA = 8
    # Costo operador SIG junior Colombia 2026
    # Ref: rangos salariales SENA/DANE tecnologos Geomatica (200k-350k/dia)
    COSTO_OPERADOR_DIA_COP = 250_000

    dias_manual  = area_total_ha / PRODUCTIVIDAD_MANUAL_HA_DIA
    costo_manual = dias_manual * COSTO_OPERADOR_DIA_COP

    # Pipeline DL: inferencia ArcGIS Pro + vectorizacion
    horas_auto = 4   # estimacion conservadora (medicion pendiente)
    dias_auto  = horas_auto / 8
    costo_auto = dias_auto * COSTO_OPERADOR_DIA_COP  # solo supervision

    # QA post-automatizacion: revision del ~9-12% de errores estimados
    FRACCION_QA = 0.25  # 25% del tiempo manual = revision rapida borrador DL
    dias_qa     = dias_manual * FRACCION_QA
    costo_qa    = dias_qa * COSTO_OPERADOR_DIA_COP

    factor_tiempo = dias_manual / (dias_auto + dias_qa)

    print('=== COMPARACION: AUTOMATIZADO vs MANUAL ===')
    print()
    print(f'Area cartografiada : {area_total_ha:.1f} ha (Construccion + Bosque)')
    print()
    col1, col2, col3, col4 = 38, 15, 11, 13
    header = f'{'Metrica':<{col1}} {'DL borrador':>{col2}} {'DL+QA':>{col3}} {'Manual':>{col4}}'
    print(header)
    print('-' * (col1+col2+col3+col4+3))
    print(f'{'Tiempo produccion (dias)':<{col1}} {dias_auto:>{col2}.1f} {'':>{col3}} {dias_manual:>{col4}.1f}')
    print(f'{'Tiempo QA/supervision (dias)':<{col1}} {'':>{col2}} {dias_qa:>{col3}.1f} {'(incluido)':>{col4}}')
    print(f'{'Tiempo TOTAL (dias)':<{col1}} {dias_auto:>{col2}.1f} {dias_auto+dias_qa:>{col3}.1f} {dias_manual:>{col4}.1f}')
    print(f'{'Costo estimado COP':<{col1}} {costo_auto:>{col2},.0f} {costo_auto+costo_qa:>{col3},.0f} {costo_manual:>{col4},.0f}')
    print(f'{'Reduccion (DL+QA vs Manual)':<{col1}} {'':>{col2}} {f"x{factor_tiempo:.0f}":>{col3}} {'(ref)':>{col4}}')
    print(f'{'Clases procesadas':<{col1}} {'2 (C+B)':>{col2}} {'2 (C+B)':>{col3}} {'todas':>{col4}}')
    print(f'{'Reproducibilidad':<{col1}} {'100% script':>{col2}} {'100% script':>{col3}} {'Variable':>{col4}}')
    print()
    print('Supuestos y referencias:')
    print(f'  * Productividad manual: {PRODUCTIVIDAD_MANUAL_HA_DIA} ha/dia (rango 5-15 ha/dia)')
    print('    Ref: DNP (2021) Guia formulacion cartografia basica oficial.')
    print(f'  * Costo operador: COP ${COSTO_OPERADOR_DIA_COP:,}/dia (SIG junior, Colombia 2026)')
    print('    Ref: rangos salariales SENA/DANE tecnologos Geomatica.')
    print(f'  * QA post-DL estimado en {int(FRACCION_QA*100)}% del tiempo manual')
    print('    (revision poligonos con errores estimados; sin visita campo).')
    print('  * Ambos productos deben pasar control de calidad IGAC (Res. 471/2020).')
    print('  * Proxy calidad ML supera umbral de utilidad practica ISPRS:')
    print('    completitud >= 70%, correccion >= 85%')
    print('    Ref: Rottensteiner et al. (2014) ISPRS JPRS, 93, 256-271.')

    # ── Figura comparativa ────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=120)

    metodos = ['DL borrador', 'DL + QA', 'Manual']
    tiempos = [dias_auto, dias_auto + dias_qa, dias_manual]
    costos  = [costo_auto/1e6, (costo_auto+costo_qa)/1e6, costo_manual/1e6]
    colores = ['#2196F3', '#64B5F6', '#FF9800']

    axes[0].bar(metodos, tiempos, color=colores, edgecolor='black', linewidth=0.8)
    axes[0].set_ylabel('Dias')
    axes[0].set_title('Tiempo de procesamiento', fontweight='bold')
    for i, v in enumerate(tiempos):
        axes[0].text(i, v + 0.3, f'{v:.1f} d', ha='center', fontweight='bold', fontsize=9)

    axes[1].bar(metodos, costos, color=colores, edgecolor='black', linewidth=0.8)
    axes[1].set_ylabel('Millones COP')
    axes[1].set_title('Costo estimado (operador)', fontweight='bold')
    for i, v in enumerate(costos):
        axes[1].text(i, v + 0.02, f'${v:.2f}M', ha='center', fontweight='bold', fontsize=9)

    for ax in axes:
        ax.set_facecolor('#F9F9F9')
        ax.grid(axis='y', alpha=0.4)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.suptitle(
        f'Comparacion metodologica — {MUNICIPIO.capitalize()}'
        f'Area: {area_total_ha:.1f} ha | Clases: Construccion + Bosque',
        fontsize=11, fontweight='bold'
    )
    plt.tight_layout()
    fig_path = Path(OUTDIR) / f'comparacion_metodologica_{MUNICIPIO}.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figura guardada: {fig_path}')


---
## 7. Resumen final para el documento

In [ ]:
import json
from pathlib import Path

json_path = Path(OUTDIR) / f'estadisticas_vectores_{MUNICIPIO}.json'
if not json_path.exists():
    print('Ejecutar primero la vectorizacion.')
else:
    with open(json_path, encoding='utf-8') as f:
        stats = json.load(f)

    print('=== RESUMEN PARA CAPITULO 4 ===')
    print()
    print('Tabla X. Resultados del pipeline de vectorizacion automatica')
    print(f'{"Clase":<20} {"Polig. raw":>12} {"Polig. final":>13} {"Area (ha)":>11} {"OGC valido":>12}')
    print('-' * 72)

    for clase, s in stats.items():
        ok = 'Si' if s['errores_ogc'] == 0 else f'No ({s["errores_ogc"]} err)'
        print(f'{clase:<20} {s["n_poligonos_raw"]:>12,} '
              f'{s["n_poligonos_final"]:>13,} '
              f'{s["area_total_m2"]/1e4:>11.1f} '
              f'{ok:>12}')

    print()
    print('Parametros del pipeline:')
    for clase, s in stats.items():
        print(f'  {clase}: RDP tol={s["simplify_tol_m"]}m | area_min={s["min_area_m2"]}m2 | hole_min={s["min_hole_m2"]}m2')

    print()
    print('Archivos de salida:')
    for clase, s in stats.items():
        print(f'  {Path(s["gpkg_path"]).name}')

    # Guardar resumen adicional para el doc
    resumen = {
        'municipio':     MUNICIPIO,
        'modelo':        'deeplabv3_resnet50_baseline',
        'clases':        list(stats.keys()),
        'total_poligonos': sum(s['n_poligonos_final'] for s in stats.values()),
        'area_total_ha': round(sum(s['area_total_m2'] for s in stats.values()) / 1e4, 2),
        'ogc_100pct':    all(s['errores_ogc'] == 0 for s in stats.values()),
        'detalles':      stats,
    }

    resumen_path = Path(OUTDIR) / f'resumen_vectorizacion_{MUNICIPIO}.json'
    with open(resumen_path, 'w', encoding='utf-8') as f:
        json.dump(resumen, f, indent=2, ensure_ascii=False)
    print(f'\nResumen guardado: {resumen_path}')